In [0]:
%run "./libraries"

In [0]:
import kagglehub
import pandas as pd
import numpy as np
import os
from xgboost import XGBRegressor
import shap
import matplotlib.pyplot as plt

In [0]:
path = kagglehub.dataset_download("sebastianwillmann/beverage-sales")

print("Dataset stored at:", path)

# Find CSV file
files = os.listdir(path)
csv_file = [f for f in files if f.endswith(".csv")][0]
csv_path = os.path.join(path, csv_file)

# Load data
df = pd.read_csv(csv_path)
#print(df.head())

# Ingest pandas DataFrame into Spark DataFrame
spark_df = spark.createDataFrame(df)
# Print sample rows from Spark DataFrame
spark_df.show(5)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Customer_Type", StringType(), True),
    StructField("Product", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Unit_Price", DoubleType(), True),
    StructField("Quantity", LongType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Total_Price", DoubleType(), True),
    StructField("Region", StringType(), True),
    StructField("Order_Date", StringType(), True)
])

print(schema)

In [0]:
# Read the beverage sales CSV dataset into Spark DataFrame using the defined schema
spark_df = spark.read.csv(spark_df, schema=schema, header=True)
spark_df.show(5)  # Display first 5 rows for quick inspection

In [0]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df = df.sort_values('Order_Date')

In [0]:
df['lag_1'] = df['Total_Price'].shift(1)
df['lag_7'] = df['Total_Price'].shift(7)
df['roll_7'] = df['Total_Price'].rolling(window=7).mean()

df = df.dropna()

In [0]:
train = df.iloc[:-30]
test = df.iloc[-30:]

X_train = train[['lag_1', 'lag_7', 'roll_7']]
y_train = train['Total_Price']

X_test = test[['lag_1', 'lag_7', 'roll_7']]
y_test = test['Total_Price']


In [0]:
model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print("Forecast for next 30 days:")
print(preds)

In [0]:
explainer = shap.Explainer(model)
shap_values = explainer(X_train)

In [0]:
shap.plots.bar(shap_values)

In [0]:
shap.initjs()
shap.force_plot(
    explainer.expected_value,
    shap_values.values[0],
    X_train.iloc[0]
)


In [0]:
# Ingest the processed DataFrame into a Delta table in Databricks
from pyspark.sql import SparkSession

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Save as managed Delta table (change name as needed)
spark_df.write.format("delta").mode("overwrite").saveAsTable("beverage_sales")

print("Ingestion complete. Table name: beverage_sales")